# Phase 2A Diagnostics — 5 Tests

**Purpose:** address 4 corrections from Round 14 review:
1. F6E ablation (legitimacy of F6E dominance)
2. Lasso stability (sensitivity to C, seed)
3. Leak-free calibration (temporal split inside train, OOT untouched)
4. LightGBM baseline (dual-track LR + LightGBM that was missing)
5. Score-discrimination stress-test design (no execution; Phase 3/4 work)

**Anchor numbers from Phase 2A initial run:**
- Final-model OOT AUC = 0.859, Gini = 0.718, Brier = 0.0082
- Final feature set: 22 features; 17/22 are F6E synthetic-bureau
- F6D negative controls: 0 survived (PASS)
- OOT calibration drift: mean predicted 1.63% vs observed 0.85% (1.92×)

In [1]:
import sys, time, json, joblib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.governance import filter_pd_eligible, get_meta_columns
from src.modeling import (
    run_stage1_prescreening, run_lasso_selection,
    fit_logit_statsmodels, compute_vif,
)
from src.evaluation import (
    compute_gini, compute_ks, compute_brier, compute_calibration_metrics,
)
from src.calibration import (
    make_calibration_split, fit_platt, fit_isotonic, apply_calibrator,
    StressTestPlan, perturb_to_target_gini,
)
np.random.seed(42)

ABT_PATH     = REPO_ROOT / "artifacts/phase2_rerun_v2/modeling_abt.parquet"
CATALOG_PATH = REPO_ROOT / "artifacts/phase2_rerun_v2/modeling_feature_catalog.csv"
OUT_DIR      = REPO_ROOT / "artifacts/phase2_rerun_v2"
DIAG_DIR     = OUT_DIR / "diagnostics"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

T0 = time.time()
catalog = pd.read_csv(CATALOG_PATH)
all_pd_eligible = filter_pd_eligible(catalog)
print(f"PD-eligible pool: {len(all_pd_eligible)}")

PD-eligible pool: 435


## Test 1 — F6E ablation

Run the same Stage 1 → Stage 2 → Stage 3 pipeline excluding all F6E (and F6E-
derived) features. Goal: quantify whether discrimination is driven by F6E
synthetic-bureau features or holds without them.

In [2]:
# Identify F6E features in the catalog
f6e_cat = catalog[catalog["feature_family"] == "F6E"]["feature_name"].tolist()
print(f"F6E features in catalog: {len(f6e_cat)}")

# PD-eligible WITHOUT any F6E
non_f6e_eligible = [f for f in all_pd_eligible if f not in set(f6e_cat)]
print(f"Non-F6E PD-eligible features: {len(non_f6e_eligible)}")

# Load only those columns + meta
df_nof6e = pd.read_parquet(ABT_PATH, columns=non_f6e_eligible + get_meta_columns())
print(f"Loaded shape: {df_nof6e.shape}")
print(f"Train DR: {df_nof6e.loc[df_nof6e['split']=='train', 'default_flag_12m'].mean():.4%}")
print(f"OOT DR  : {df_nof6e.loc[df_nof6e['split']=='oot',   'default_flag_12m'].mean():.4%}")

F6E features in catalog: 200
Non-F6E PD-eligible features: 235


Loaded shape: (534314, 239)
Train DR: 1.6330%
OOT DR  : 0.8474%


In [3]:
# Stage 1 (no caching for ablation since features differ)
t = time.time()
stage1_nof6e = run_stage1_prescreening(df_nof6e, "default_flag_12m", non_f6e_eligible)
print(f"Stage 1 (no F6E) wall: {time.time()-t:.1f}s, survivors: {len(stage1_nof6e)}")

Stage 1 (no F6E) wall: 26.2s, survivors: 66


In [4]:
# Stage 2 — Lasso single-fit at C=0.05
t = time.time()
stage2_nof6e = run_lasso_selection(
    df_nof6e, stage1_nof6e["feature"].tolist(), "default_flag_12m",
    use_cv=False, fixed_c=0.05, train_subsample=100_000, random_state=42,
)
print(f"Stage 2 (no F6E) wall: {time.time()-t:.1f}s, survivors: {len(stage2_nof6e)}")

Stage 2 (no F6E) wall: 11.7s, survivors: 34


In [5]:
# Stage 3 — VIF + p-value filter + final fit
final_features_nof6e = list(stage2_nof6e)
vif_df = compute_vif(df_nof6e, final_features_nof6e)
high_vif = vif_df.loc[vif_df["vif"] > 10, "feature"].tolist()
final_features_nof6e = [f for f in final_features_nof6e if f not in high_vif]
print(f"After VIF filter: {len(final_features_nof6e)} features (dropped {len(high_vif)} high-VIF)")

model_nof6e = fit_logit_statsmodels(df_nof6e, final_features_nof6e, "default_flag_12m")
significant = model_nof6e.pvalues[model_nof6e.pvalues < 0.05].index.tolist()
final_features_nof6e = [f for f in significant if f != "const"]
print(f"After p<0.05 filter: {len(final_features_nof6e)} features")
final_model_nof6e = fit_logit_statsmodels(df_nof6e, final_features_nof6e, "default_flag_12m")
print("\nFinal-model coefficients:")
for f in final_features_nof6e:
    print(f"  {f:<45}  coef={final_model_nof6e.params[f]:+.5f}  p={final_model_nof6e.pvalues[f]:.4g}")

After VIF filter: 11 features (dropped 23 high-VIF)


After p<0.05 filter: 7 features



Final-model coefficients:
  app_nom_job_code                               coef=-1.13537  p=0
  act_age_log1p                                  coef=-5.08540  p=8.633e-256
  mean_cars_by_job_code                          coef=+35.62502  p=1.677e-59
  app_nom_marital_status                         coef=-0.61768  p=0
  median_income_by_job_code                      coef=-0.00023  p=3.374e-157
  app_nom_branch                                 coef=-0.17234  p=1.876e-36
  app_nom_city                                   coef=-0.14634  p=7.256e-27


In [6]:
# Score and metrics
import statsmodels.api as sm
X_nof6e = sm.add_constant(
    df_nof6e[final_features_nof6e].fillna(df_nof6e[final_features_nof6e].median(numeric_only=True)),
    has_constant="add",
)
df_nof6e["pd_score_nof6e"] = final_model_nof6e.predict(X_nof6e)
tr = df_nof6e["split"]=="train"; oot = df_nof6e["split"]=="oot"
metrics_nof6e = {
    "train": {
        "auc"  : (compute_gini(df_nof6e.loc[tr,"default_flag_12m"], df_nof6e.loc[tr,"pd_score_nof6e"]) + 1)/2,
        "gini" : compute_gini(df_nof6e.loc[tr,"default_flag_12m"], df_nof6e.loc[tr,"pd_score_nof6e"]),
        "ks"   : compute_ks  (df_nof6e.loc[tr,"default_flag_12m"], df_nof6e.loc[tr,"pd_score_nof6e"]),
        "brier": compute_brier(df_nof6e.loc[tr,"default_flag_12m"], df_nof6e.loc[tr,"pd_score_nof6e"]),
    },
    "oot": {
        "auc"  : (compute_gini(df_nof6e.loc[oot,"default_flag_12m"], df_nof6e.loc[oot,"pd_score_nof6e"]) + 1)/2,
        "gini" : compute_gini(df_nof6e.loc[oot,"default_flag_12m"], df_nof6e.loc[oot,"pd_score_nof6e"]),
        "ks"   : compute_ks  (df_nof6e.loc[oot,"default_flag_12m"], df_nof6e.loc[oot,"pd_score_nof6e"]),
        "brier": compute_brier(df_nof6e.loc[oot,"default_flag_12m"], df_nof6e.loc[oot,"pd_score_nof6e"]),
    },
}
print("\n=== Test 1 result ===")
print(json.dumps(metrics_nof6e, indent=2))

# Compare to current full-F6E model
with open(OUT_DIR / "model_metrics.json") as f:
    metrics_full = json.load(f)
print("\n=== Comparison: full-F6E vs no-F6E ===")
comp = pd.DataFrame({
    "full_F6E_train": metrics_full["train"], "no_F6E_train": metrics_nof6e["train"],
    "full_F6E_oot":   metrics_full["oot"],   "no_F6E_oot":   metrics_nof6e["oot"],
})
print(comp.round(4).to_string())

# Calibration on OOT (raw, uncalibrated)
cal_nof6e_oot = compute_calibration_metrics(
    df_nof6e.loc[oot, "default_flag_12m"], df_nof6e.loc[oot, "pd_score_nof6e"]
)
print(f"\nNo-F6E OOT calibration: ECE={cal_nof6e_oot.get('ece', float('nan')):.4f}, "
      f"mean_pred={cal_nof6e_oot.get('mean_pred', float('nan')):.4%}, "
      f"base_rate={cal_nof6e_oot.get('base_rate', float('nan')):.4%}")

# Save
test1_out = {
    "n_features_pool": len(non_f6e_eligible),
    "n_stage1": len(stage1_nof6e),
    "n_stage2": len(stage2_nof6e),
    "n_final":  len(final_features_nof6e),
    "final_features": final_features_nof6e,
    "metrics": metrics_nof6e,
    "calib_oot_uncal": {k: float(v) if isinstance(v, (int,float)) else v
                        for k, v in cal_nof6e_oot.items()},
}
with open(DIAG_DIR / "test1_f6e_ablation.json", "w") as f:
    json.dump(test1_out, f, indent=2, default=str)
print("\nSaved: diagnostics/test1_f6e_ablation.json")


=== Test 1 result ===
{
  "train": {
    "auc": 0.7938758898346447,
    "gini": 0.5877517796692895,
    "ks": 0.4518301201517804,
    "brier": 0.015378107686033909
  },
  "oot": {
    "auc": 0.8361300713104687,
    "gini": 0.6722601426209374,
    "ks": 0.5394825111384536,
    "brier": 0.008207038913770828
  }
}

=== Comparison: full-F6E vs no-F6E ===
       full_F6E_train  no_F6E_train  full_F6E_oot  no_F6E_oot
gini           0.6411        0.5878        0.7181      0.6723
ks             0.4811        0.4518        0.5475      0.5395
brier          0.0152        0.0154        0.0082      0.0082
auc            0.8205        0.7939        0.8591      0.8361

No-F6E OOT calibration: ECE=0.0078, mean_pred=1.6283%, base_rate=0.8474%

Saved: diagnostics/test1_f6e_ablation.json


## Test 2 — Lasso stability (3 C × 3 seeds)

Sensitivity of single-fit Lasso selection across (C, seed) configurations.
We use the *full-pool* Stage 1 survivors (with F6E) since the goal here is to
test the Stage 2 step's reliability, not the F6E hypothesis.

In [7]:
# Reload Stage 1 cache from full-pool run
stage1_full = joblib.load(REPO_ROOT / "artifacts/notebook_cache/stage1_results.pkl")
features_s1 = stage1_full["feature"].tolist()
print(f"Reusing Stage 1 survivors (full pool): {len(features_s1)}")

# Need full-pool df with all those columns
df_full = pd.read_parquet(ABT_PATH, columns=features_s1 + get_meta_columns())
print(f"Loaded ABT for stability test: {df_full.shape}")

Reusing Stage 1 survivors (full pool): 120


Loaded ABT for stability test: (534314, 124)


In [8]:
Cs = [0.02, 0.05, 0.10]
SEEDS = [42, 101, 202]

stability_rows = []
selected_sets = {}

for c in Cs:
    for sd in SEEDS:
        t = time.time()
        sel = run_lasso_selection(
            df_full, features_s1, "default_flag_12m",
            use_cv=False, fixed_c=c, train_subsample=100_000, random_state=sd,
        )
        wall = time.time() - t
        f6e_count = sum(1 for f in sel if f in set(catalog[catalog["feature_family"]=="F6E"]["feature_name"]))
        f6d_count = sum(1 for f in sel if f.startswith("random_"))
        stability_rows.append({
            "C": c, "seed": sd,
            "n_survivors": len(sel),
            "f6e_count": f6e_count,
            "f6e_pct": round(100*f6e_count/max(len(sel),1), 1),
            "f6d_count": f6d_count,
            "wall_sec": round(wall, 1),
        })
        selected_sets[(c, sd)] = set(sel)

stab_df = pd.DataFrame(stability_rows)
print("\nStability table (per C × seed):")
print(stab_df.to_string(index=False))


Stability table (per C × seed):
   C  seed  n_survivors  f6e_count  f6e_pct  f6d_count  wall_sec
0.02    42           45         18     40.0          0      14.1
0.02   101           46         18     39.1          0      15.0
0.02   202           39         11     28.2          0      23.9
0.05    42           64         33     51.6          0      17.6
0.05   101           59         27     45.8          0      11.7
0.05   202           54         22     40.7          0      28.7
0.10    42           72         40     55.6          0      33.4
0.10   101           66         33     50.0          0      29.3
0.10   202           64         30     46.9          0      77.3


In [9]:
# Pairwise Jaccard overlap
keys = list(selected_sets.keys())
jaccard = pd.DataFrame(0.0, index=[str(k) for k in keys], columns=[str(k) for k in keys])
for i, ki in enumerate(keys):
    for j, kj in enumerate(keys):
        a, b = selected_sets[ki], selected_sets[kj]
        u = a | b
        jaccard.iloc[i, j] = len(a & b) / max(len(u), 1)
print("\nPairwise Jaccard overlap of selected feature sets:")
print(jaccard.round(3).to_string())

# Frequency of each feature across all 9 runs
all_selected = pd.Series([f for s in selected_sets.values() for f in s])
freq = all_selected.value_counts()
core = freq[freq >= 8].index.tolist()  # selected in 8/9+ runs
sometimes = freq[(freq >= 5) & (freq < 8)].index.tolist()
print(f"\nFeatures selected in >=8/9 runs (core, robust): {len(core)}")
for f in core[:30]:
    print(f"  {f:<50}  count={freq[f]}")
print(f"\nFeatures selected in 5-7/9 runs (boundary): {len(sometimes)}")


Pairwise Jaccard overlap of selected feature sets:
             (0.02, 42)  (0.02, 101)  (0.02, 202)  (0.05, 42)  (0.05, 101)  (0.05, 202)  (0.1, 42)  (0.1, 101)  (0.1, 202)
(0.02, 42)        1.000        0.655        0.556       0.627        0.625        0.523      0.481       0.542       0.493
(0.02, 101)       0.655        1.000        0.667       0.486        0.780        0.538      0.439       0.623       0.528
(0.02, 202)       0.556        0.667        1.000       0.431        0.531        0.661      0.388       0.419       0.585
(0.05, 42)        0.627        0.486        0.431       1.000        0.519        0.513      0.789       0.566       0.524
(0.05, 101)       0.625        0.780        0.531       0.519        1.000        0.507      0.489       0.812       0.519
(0.05, 202)       0.523        0.538        0.661       0.513        0.507        1.000      0.518       0.500       0.844
(0.1, 42)         0.481        0.439        0.388       0.789        0.489        0.518

In [10]:
# Save (cast pandas/numpy ints to Python ints for JSON safety)
test2_out = {
    "configs": stability_rows,
    "jaccard_min": float(jaccard.replace(1.0, np.nan).min().min()),
    "jaccard_mean": float(jaccard.replace(1.0, np.nan).mean().mean()),
    "core_features_8plus_of_9": core,
    "boundary_features_5_to_7": sometimes,
    "f6e_pct_range": [float(stab_df["f6e_pct"].min()), float(stab_df["f6e_pct"].max())],
    "f6d_count_range": [int(stab_df["f6d_count"].min()), int(stab_df["f6d_count"].max())],
}
with open(DIAG_DIR / "test2_lasso_stability.json", "w") as f:
    json.dump(test2_out, f, indent=2, default=str)
print(f"\nJaccard min={test2_out['jaccard_min']:.3f}, mean={test2_out['jaccard_mean']:.3f}")
print(f"F6E share across configs: {stab_df['f6e_pct'].min()}-{stab_df['f6e_pct'].max()}%")
print(f"F6D survivors across configs: {stab_df['f6d_count'].min()}-{stab_df['f6d_count'].max()} (target=0)")
print("Saved: diagnostics/test2_lasso_stability.json")


Jaccard min=0.388, mean=0.567
F6E share across configs: 28.2-55.6%
F6D survivors across configs: 0-0 (target=0)
Saved: diagnostics/test2_lasso_stability.json


## Test 3 — Leak-free calibration

**Splits:**
- model training: cohorts 202509-202610 (14 cohorts)
- calibration: cohorts 202611, 202612 (last 2 train cohorts)
- OOT: cohorts 202701-202706 (UNTOUCHED for calibration fitting)

Refit final-feature LR on the reduced training set, score everything, then fit
Platt and isotonic calibrators on the calibration cohorts. Evaluate calibrated
PD on OOT.

In [11]:
# Load full-pool df + final-model features from Phase 2A
final_model = joblib.load(OUT_DIR / "final_model.pkl")
final_features = pd.read_csv(OUT_DIR / "final_feature_set.csv")["feature"].tolist()
print(f"Reusing final feature set ({len(final_features)} features)")

df_cal = pd.read_parquet(ABT_PATH,
                          columns=final_features + get_meta_columns())
print(f"Loaded ABT for calibration test: {df_cal.shape}")

# Define splits
TRAIN_FOR_MODEL = list(range(202509, 202613))  # 202509..202612 inclusive
TRAIN_FOR_MODEL = [p for p in TRAIN_FOR_MODEL if p % 100 in range(1, 13)]
TRAIN_FOR_MODEL = [p for p in TRAIN_FOR_MODEL if p <= 202610]  # exclude calib cohorts
CALIB_PERIODS = [202611, 202612]
print(f"Train-for-model cohorts: {TRAIN_FOR_MODEL}")
print(f"Calibration cohorts:    {CALIB_PERIODS}")

split_new = make_calibration_split(df_cal, TRAIN_FOR_MODEL, CALIB_PERIODS)
df_cal["split_new"] = split_new
print("\nNew split counts:")
print(df_cal["split_new"].value_counts().to_string())
print("\nDR by new split:")
for s in ["train_for_model", "calib", "oot"]:
    sub = df_cal[df_cal["split_new"]==s]
    if len(sub) > 0:
        print(f"  {s:<18} n={len(sub):>7,}  DR={(sub['default_flag_12m']==1).mean():.4%}  "
              f"events={int((sub['default_flag_12m']==1).sum())}")

Reusing final feature set (22 features)


Loaded ABT for calibration test: (534314, 26)
Train-for-model cohorts: [202509, 202510, 202511, 202512, 202601, 202602, 202603, 202604, 202605, 202606, 202607, 202608, 202609, 202610]
Calibration cohorts:    [202611, 202612]

New split counts:
split_new
train_for_model    340727
oot                144789
calib               48798

DR by new split:
  train_for_model    n=340,727  DR=1.7345%  events=5910
  calib              n= 48,798  DR=0.9242%  events=451


  oot                n=144,789  DR=0.8474%  events=1227


In [12]:
# Refit LR on TRAIN_FOR_MODEL only
import statsmodels.api as sm
mask_tfm = df_cal["split_new"] == "train_for_model"
X_tfm = df_cal.loc[mask_tfm, final_features].fillna(df_cal[final_features].median(numeric_only=True))
y_tfm = df_cal.loc[mask_tfm, "default_flag_12m"].astype(int)
X_tfm = sm.add_constant(X_tfm, has_constant="add")
model_calib = sm.Logit(y_tfm, X_tfm).fit(disp=False, maxiter=200)
print(f"Refit LR on {mask_tfm.sum():,} train_for_model rows, {len(final_features)} features")

# Score everything
X_full = sm.add_constant(
    df_cal[final_features].fillna(df_cal[final_features].median(numeric_only=True)),
    has_constant="add",
)
df_cal["pd_raw"] = model_calib.predict(X_full)

# Fit calibrators on calib split only
mask_calib = df_cal["split_new"] == "calib"
y_calib = df_cal.loc[mask_calib, "default_flag_12m"].astype(int).to_numpy()
s_calib = df_cal.loc[mask_calib, "pd_raw"].to_numpy()
print(f"Fitting calibrators on {len(s_calib):,} calib rows, {y_calib.sum()} events")

platt = fit_platt(s_calib, y_calib)
iso   = fit_isotonic(s_calib, y_calib)

df_cal["pd_platt"] = apply_calibrator(df_cal["pd_raw"].to_numpy(), platt, "platt")
df_cal["pd_iso"]   = apply_calibrator(df_cal["pd_raw"].to_numpy(), iso, "isotonic")

# Save calibrators for later
joblib.dump({"platt": platt, "isotonic": iso,
             "calib_periods": CALIB_PERIODS,
             "train_for_model_periods": TRAIN_FOR_MODEL,
             "final_features": final_features},
            DIAG_DIR / "calibrators.pkl")
print("Saved: diagnostics/calibrators.pkl")

Refit LR on 340,727 train_for_model rows, 22 features


Fitting calibrators on 48,798 calib rows, 451 events
Saved: diagnostics/calibrators.pkl


In [13]:
# Evaluate on OOT (untouched by calibration fitting)
mask_oot = df_cal["split_new"] == "oot"
y_oot = df_cal.loc[mask_oot, "default_flag_12m"].astype(int).to_numpy()
results = {}
for kind in ["pd_raw", "pd_platt", "pd_iso"]:
    s = df_cal.loc[mask_oot, kind].to_numpy()
    cal = compute_calibration_metrics(y_oot, s)
    results[kind] = {
        "auc": (compute_gini(y_oot, s) + 1) / 2,
        "gini": compute_gini(y_oot, s),
        "ks": compute_ks(y_oot, s),
        "brier": compute_brier(y_oot, s),
        "ece": cal.get("ece", float("nan")),
        "mean_pred": cal.get("mean_pred", float("nan")),
        "base_rate": cal.get("base_rate", float("nan")),
    }

print("=== Test 3 OOT metrics ===")
print(pd.DataFrame(results).round(5).to_string())

# Save
with open(DIAG_DIR / "test3_calibration.json", "w") as f:
    json.dump({
        "splits": {
            "train_for_model_periods": TRAIN_FOR_MODEL,
            "calib_periods": CALIB_PERIODS,
            "n_train_for_model": int(mask_tfm.sum()),
            "n_calib": int(mask_calib.sum()),
            "n_oot": int(mask_oot.sum()),
        },
        "oot_metrics_by_calibration": results,
    }, f, indent=2, default=str)
print("\nSaved: diagnostics/test3_calibration.json")

=== Test 3 OOT metrics ===
            pd_raw  pd_platt   pd_iso
auc        0.85909   0.85909  0.85732
gini       0.71817   0.71817  0.71464
ks         0.54837   0.54837  0.54352
brier      0.00821   0.00799  0.00800
ece        0.00879   0.00107  0.00079
mean_pred  0.01727   0.00921  0.00926
base_rate  0.00847   0.00847  0.00847

Saved: diagnostics/test3_calibration.json


## Test 4 — LightGBM baseline (dual-track)

Train LightGBM on **catalog-filtered PD-eligible raw features**, with native
categorical handling and `scale_pos_weight = n_neg / n_pos`. Same temporal
calibration split as Test 3 for leak-free calibration.

In [14]:
import lightgbm as lgb

# Use the SAME PD-eligible pool (with F6E) for fair LightGBM benchmark
df_lgb = pd.read_parquet(ABT_PATH, columns=all_pd_eligible + get_meta_columns())
df_lgb["split_new"] = make_calibration_split(df_lgb, TRAIN_FOR_MODEL, CALIB_PERIODS)
print(f"LightGBM df: {df_lgb.shape}")

# Identify categorical features by catalog (CATEGORICAL_SAFE in the original taxonomy)
CAT_FEATURES = ["app_nom_branch", "app_nom_gender", "app_nom_job_code",
                "app_nom_marital_status", "app_nom_city", "app_nom_home_status"]
cat_in_pool = [c for c in CAT_FEATURES if c in all_pd_eligible]
print(f"Categorical features in pool: {cat_in_pool}")

# Cast cats to pandas Categorical for native handling
for c in cat_in_pool:
    df_lgb[c] = df_lgb[c].astype("category")

mask_tfm  = df_lgb["split_new"] == "train_for_model"
mask_calib= df_lgb["split_new"] == "calib"
mask_oot  = df_lgb["split_new"] == "oot"

X_tr = df_lgb.loc[mask_tfm, all_pd_eligible]
y_tr = df_lgb.loc[mask_tfm, "default_flag_12m"].astype(int).to_numpy()
n_pos = int(y_tr.sum()); n_neg = int(len(y_tr) - n_pos)
spw = n_neg / max(n_pos, 1)
print(f"Train: {len(y_tr):,}  pos={n_pos}  neg={n_neg}  scale_pos_weight={spw:.2f}")

LightGBM df: (534314, 440)
Categorical features in pool: ['app_nom_branch', 'app_nom_gender', 'app_nom_job_code', 'app_nom_marital_status', 'app_nom_city', 'app_nom_home_status']


Train: 340,727  pos=5910  neg=334817  scale_pos_weight=56.65


In [15]:
# Use a small validation slice from train for early stopping
from sklearn.model_selection import train_test_split
X_tr_in, X_val, y_tr_in, y_val = train_test_split(
    X_tr, y_tr, test_size=0.15, stratify=y_tr, random_state=42,
)
print(f"Inner train: {len(y_tr_in):,}  Val: {len(y_val):,}")

t = time.time()
lgbm = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=200,
    reg_alpha=0.0, reg_lambda=1.0,
    colsample_bytree=0.7,
    subsample=0.7, subsample_freq=1,
    scale_pos_weight=spw,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
lgbm.fit(
    X_tr_in, y_tr_in,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)],
    categorical_feature=cat_in_pool,
)
print(f"LightGBM wall: {time.time()-t:.1f}s, best_iter={lgbm.best_iteration_}")

Inner train: 289,617  Val: 51,110


LightGBM wall: 6.8s, best_iter=1


In [16]:
# Score everything
df_lgb["pd_lgb_raw"] = lgbm.predict_proba(df_lgb[all_pd_eligible])[:, 1]

# Calibrate on the calib slice (NEVER on OOT)
y_calib = df_lgb.loc[mask_calib, "default_flag_12m"].astype(int).to_numpy()
s_calib = df_lgb.loc[mask_calib, "pd_lgb_raw"].to_numpy()
platt_lgb = fit_platt(s_calib, y_calib)
iso_lgb   = fit_isotonic(s_calib, y_calib)
df_lgb["pd_lgb_platt"] = apply_calibrator(df_lgb["pd_lgb_raw"].to_numpy(), platt_lgb, "platt")
df_lgb["pd_lgb_iso"]   = apply_calibrator(df_lgb["pd_lgb_raw"].to_numpy(), iso_lgb, "isotonic")

# Metrics on OOT
y_oot = df_lgb.loc[mask_oot, "default_flag_12m"].astype(int).to_numpy()
lgb_results = {}
for kind in ["pd_lgb_raw", "pd_lgb_platt", "pd_lgb_iso"]:
    s = df_lgb.loc[mask_oot, kind].to_numpy()
    cal = compute_calibration_metrics(y_oot, s)
    lgb_results[kind] = {
        "auc": (compute_gini(y_oot, s) + 1) / 2,
        "gini": compute_gini(y_oot, s),
        "ks": compute_ks(y_oot, s),
        "brier": compute_brier(y_oot, s),
        "ece": cal.get("ece", float("nan")),
        "mean_pred": cal.get("mean_pred", float("nan")),
        "base_rate": cal.get("base_rate", float("nan")),
    }
print("=== LightGBM OOT metrics ===")
print(pd.DataFrame(lgb_results).round(5).to_string())

=== LightGBM OOT metrics ===
           pd_lgb_raw  pd_lgb_platt  pd_lgb_iso
auc           0.87262       0.87262     0.87040
gini          0.74525       0.74525     0.74080
ks            0.59611       0.59611     0.59611
brier         0.01099       0.00810     0.00809
ece           0.04492       0.00095     0.00084
mean_pred     0.05339       0.00938     0.00929
base_rate     0.00847       0.00847     0.00847


In [17]:
# Top 20 feature importances
imp_df = pd.DataFrame({
    "feature": all_pd_eligible,
    "importance_split": lgbm.booster_.feature_importance(importance_type="split"),
    "importance_gain":  lgbm.booster_.feature_importance(importance_type="gain"),
}).sort_values("importance_gain", ascending=False).head(20)
print("\nTop 20 LightGBM features by gain:")
print(imp_df.to_string(index=False))

# F6D check: any random_* in top 20?
f6d_in_top = [f for f in imp_df["feature"] if f.startswith("random_")]
print(f"\nF6D negative controls in top 20: {len(f6d_in_top)}")
if f6d_in_top:
    print(f"  Features: {f6d_in_top}")
else:
    print("  IDEAL: no random_* in top 20")


Top 20 LightGBM features by gain:
                   feature  importance_split  importance_gain
         job_code_x_income                 1    750759.000000
          age_x_n_children                 5    462439.830078
                   act_age                 5    340386.220215
   count_by_marital_status                 2    237152.101562
            app_nom_gender                 3    160303.500000
mean_age_by_marital_status                 2     52034.400391
synth_self_reported_income                 1     27663.699219
       synth_secured_limit                 1     19424.000000
       synth_highest_limit                 1     14785.200195
 synth_int_balance_x_score                 1      8894.320312
     synth_revolving_limit                 1      7941.419922
 synth_int_inc_x_nchildren                 1      7693.029785
    synth_available_credit                 1      7621.359863
   synth_auto_loan_balance                 1      7273.120117
 synth_household_income_v1         

In [18]:
# Save model + results
joblib.dump({"model": lgbm, "best_iteration": lgbm.best_iteration_,
             "scale_pos_weight": spw, "categorical_features": cat_in_pool,
             "platt": platt_lgb, "isotonic": iso_lgb,
             "feature_list": all_pd_eligible},
            REPO_ROOT / "artifacts/pd_model/lightgbm_model.pkl",
            )

test4_out = {
    "n_features": len(all_pd_eligible),
    "best_iter": int(lgbm.best_iteration_),
    "scale_pos_weight": float(spw),
    "oot_metrics": lgb_results,
    "top20_features": imp_df.to_dict(orient="records"),
    "f6d_in_top20": f6d_in_top,
}
with open(DIAG_DIR / "test4_lightgbm.json", "w") as f:
    json.dump(test4_out, f, indent=2, default=str)
print(f"\nSaved: diagnostics/test4_lightgbm.json")
print(f"Saved: artifacts/pd_model/lightgbm_model.pkl")


Saved: diagnostics/test4_lightgbm.json
Saved: artifacts/pd_model/lightgbm_model.pkl


## Test 5 — Score-discrimination stress-test design (no execution)

Design only — Phase 3/4 will execute. Generates calibrated PD score variants
at controlled Gini levels (0.30, 0.45, 0.60, raw) so the profit cut-off
analysis can compare regimes that isolate discrimination from calibration.

In [19]:
plan = StressTestPlan()
print("StressTestPlan:")
print(f"  target_ginis     : {plan.target_ginis}")
print(f"  tolerance        : {plan.tolerance}")
print(f"  base_rate_match  : {plan.base_rate_match}")
print(f"  seed             : {plan.seed}")
print(f"  notes            : {plan.notes}")

# Sanity-check: run perturbation on a small sample (10K rows) at one target
# to verify the API works, but do NOT generate the full stress-test set yet.
sample_idx = np.random.RandomState(42).choice(len(df_cal), size=10000, replace=False)
y_smp = df_cal.iloc[sample_idx]["default_flag_12m"].astype(int).to_numpy()
s_smp = df_cal.iloc[sample_idx]["pd_iso"].to_numpy()
print(f"\nSanity-check (10K sample, target_gini=0.30):")
sp, meta = perturb_to_target_gini(s_smp, y_smp, target_gini=0.30, seed=42, n_iter=20)
print(json.dumps(meta, indent=2))
print("\nFunction works. Phase 3 will run on full data.")

# Save plan + sanity output
test5_out = {
    "stress_test_plan": {
        "target_ginis": list(plan.target_ginis),
        "tolerance": plan.tolerance,
        "sigma_search_bounds": list(plan.sigma_search_bounds),
        "n_search_iter": plan.n_search_iter,
        "base_rate_match": plan.base_rate_match,
        "seed": plan.seed,
        "notes": plan.notes,
    },
    "sanity_check": meta,
    "executed": False,
    "execution_phase": "Phase 3/4",
}
with open(DIAG_DIR / "test5_stress_test_design.json", "w") as f:
    json.dump(test5_out, f, indent=2, default=str)
print("\nSaved: diagnostics/test5_stress_test_design.json")
print(f"\nTotal Phase 2A diagnostics wall: {time.time()-T0:.1f}s")

StressTestPlan:
  target_ginis     : (0.6, 0.45, 0.3)
  tolerance        : 0.005
  base_rate_match  : True
  seed             : 42
  notes            : Logit-noise perturbation; re-calibrate mean to base rate after perturbation so that profit-cutoff comparison isolates discrimination shift from calibration shift.

Sanity-check (10K sample, target_gini=0.30):
{
  "sigma": 3.0810546875,
  "achieved_gini": 0.30330410126582263,
  "target_gini": 0.3,
  "raw_gini": 0.6653974683544304,
  "tolerance_met": true,
  "re_calibrated_to_base_rate": true,
  "base_rate": 0.0125,
  "seed": 42
}

Function works. Phase 3 will run on full data.

Saved: diagnostics/test5_stress_test_design.json

Total Phase 2A diagnostics wall: 324.9s


## Summary

All 5 tests complete. Per-test deliverables saved under
`artifacts/phase2_rerun_v2/diagnostics/`:

- `test1_f6e_ablation.json`
- `test2_lasso_stability.json`
- `test3_calibration.json`
- `test4_lightgbm.json`
- `test5_stress_test_design.json`

Decision recommendation written to `phase2_modeling_report.md` (Phase 3 input).